<a href="https://colab.research.google.com/github/omeletteZ/BDomlet/blob/main/4%D0%BB%D0%B12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark findspark -q

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()
print("Spark запущен!")

Spark запущен!


In [12]:
import pandas as pd

df_pandas = pd.read_csv(
    '/content/drive/MyDrive/Colab Notebooks/result.csv',
    names=['lsn', 'action', 'event_time', 'row_id', 'name'],
    skiprows=1
)
print(f"Всего записей: {len(df_pandas)}")
df_pandas.head()

Всего записей: 25


,lsn,action,event_time,row_id,name
0,30269432,UPDATE,2026-03-20 17:50:38,100,Alice
1,30269608,DELETE,2026-03-20 17:50:43,100,Alice_updated
2,30270056,INSERT,2026-03-20 18:03:27,1,Alice
3,30271344,INSERT,2026-03-20 18:03:27,2,Bob
4,30271456,INSERT,2026-03-20 18:03:27,3,Charlie


In [13]:
df_spark = spark.createDataFrame(df_pandas)
print("Схема таблицы:")
df_spark.printSchema()
df_spark.show(5)

Схема таблицы:
root
 |-- lsn: long (nullable = true)
 |-- action: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- row_id: long (nullable = true)
 |-- name: string (nullable = true)

+--------+------+-------------------+------+-------------+
|     lsn|action|         event_time|row_id|         name|
+--------+------+-------------------+------+-------------+
|30269432|UPDATE|2026-03-20 17:50:38|   100|        Alice|
|30269608|DELETE|2026-03-20 17:50:43|   100|Alice_updated|
|30270056|INSERT|2026-03-20 18:03:27|     1|        Alice|
|30271344|INSERT|2026-03-20 18:03:27|     2|          Bob|
|30271456|INSERT|2026-03-20 18:03:27|     3|      Charlie|
+--------+------+-------------------+------+-------------+
only showing top 5 rows


Задание: map (uppercase всех имён)

In [14]:
from pyspark.sql.functions import upper

df_upper = df_spark.withColumn("name", upper(df_spark["name"]))
df_upper.show(5)

+--------+------+-------------------+------+-------------+
|     lsn|action|         event_time|row_id|         name|
+--------+------+-------------------+------+-------------+
|30269432|UPDATE|2026-03-20 17:50:38|   100|        ALICE|
|30269608|DELETE|2026-03-20 17:50:43|   100|ALICE_UPDATED|
|30270056|INSERT|2026-03-20 18:03:27|     1|        ALICE|
|30271344|INSERT|2026-03-20 18:03:27|     2|          BOB|
|30271456|INSERT|2026-03-20 18:03:27|     3|      CHARLIE|
+--------+------+-------------------+------+-------------+
only showing top 5 rows


Задание: filter (только INSERT операции)

In [15]:
df_inserts = df_spark.filter(df_spark["action"] == "INSERT")
print(f"Количество INSERT: {df_inserts.count()}")
df_inserts.show()

Количество INSERT: 7
+--------+------+-------------------+------+-------+
|     lsn|action|         event_time|row_id|   name|
+--------+------+-------------------+------+-------+
|30270056|INSERT|2026-03-20 18:03:27|     1|  Alice|
|30271344|INSERT|2026-03-20 18:03:27|     2|    Bob|
|30271456|INSERT|2026-03-20 18:03:27|     3|Charlie|
|30271576|INSERT|2026-03-20 18:03:27|     4|  Diana|
|30271696|INSERT|2026-03-20 18:03:27|     5|    Eve|
|30273288|INSERT|2026-03-20 18:03:28|     6|  Frank|
|30273408|INSERT|2026-03-20 18:03:28|     7|  Grace|
+--------+------+-------------------+------+-------+



Задание: подсчёт каждого типа операции (reduceByKey)

In [16]:
from pyspark.sql.functions import count

df_counts = df_spark.groupBy("action").agg(count("*").alias("count"))
df_counts.show()

+------+-----+
|action|count|
+------+-----+
|INSERT|    7|
|DELETE|    7|
|UPDATE|   11|
+------+-----+



То же самое через RDD

In [17]:
rdd = df_spark.rdd.map(lambda row: (row['action'], 1))
counts = rdd.reduceByKey(lambda a, b: a + b)
print("Результат через RDD:")
counts.toDF().show()

Результат через RDD:
+------+---+
|    _1| _2|
+------+---+
|UPDATE| 11|
|DELETE|  7|
|INSERT|  7|
+------+---+

